In [2]:
# ============================================================
# AI Based Intelligent RF Spectrum Signal Identification System
# Notebook: 02_data_preprocessing.ipynb
# Purpose: Preprocess RadioML 2016.10a Dataset for CNN Training
# ============================================================

import os
import pickle
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("=" * 60)
print("DATA PREPROCESSING STARTED")
print("=" * 60)


# ============================================================
# 1. DEFINE PATHS
# ============================================================

dataset_path = "../data/dataset/RML2016.10a_dict.pkl"
processed_data_dir = "../data/processed"

os.makedirs(processed_data_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"Dataset not found:\n{os.path.abspath(dataset_path)}"
    )

print("\nDataset found:")
print(os.path.abspath(dataset_path))


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("\nLoading RadioML 2016.10a dataset...")

with open(dataset_path, "rb") as file:
    dataset = pickle.load(file, encoding="latin1")

print("Dataset loaded successfully.")


# ============================================================
# 3. EXTRACT DATASET INFORMATION
# ============================================================

modulation_types = sorted(set(key[0] for key in dataset.keys()))
snr_values = sorted(set(key[1] for key in dataset.keys()))

print("\nNumber of modulation classes:", len(modulation_types))
print("Modulation classes:", modulation_types)

print("\nNumber of SNR levels:", len(snr_values))
print("SNR values:", snr_values)


# ============================================================
# 4. COMBINE ALL IQ SIGNALS AND LABELS
# ============================================================

print("\nCombining IQ signals...")

X_list = []
y_list = []
snr_list = []

for (modulation, snr), signals in dataset.items():

    X_list.append(signals)

    number_of_signals = signals.shape[0]

    y_list.extend([modulation] * number_of_signals)

    snr_list.extend([snr] * number_of_signals)


X = np.concatenate(X_list, axis=0)
y = np.array(y_list)
snr_labels = np.array(snr_list)

print("\nCombined dataset shape:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("SNR shape:", snr_labels.shape)


# ============================================================
# 5. ENCODE MODULATION LABELS
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("\nLabel encoding:")

for index, modulation in enumerate(label_encoder.classes_):
    print(f"{index} -> {modulation}")


# ============================================================
# 6. CHECK CLASS DISTRIBUTION
# ============================================================

print("\nClass distribution:")

for modulation in modulation_types:

    count = np.sum(y == modulation)

    print(f"{modulation}: {count}")


# ============================================================
# 7. NORMALIZE EACH IQ SIGNAL
# ============================================================

print("\nNormalizing IQ signals...")

# Calculate magnitude for each I/Q sample
magnitude = np.sqrt(
    X[:, 0, :] ** 2 +
    X[:, 1, :] ** 2
)

# Find maximum magnitude for each signal
max_magnitude = np.max(
    magnitude,
    axis=1,
    keepdims=True
)

# Prevent division by zero
max_magnitude[max_magnitude == 0] = 1.0

# Normalize each signal
X_normalized = X / max_magnitude[:, np.newaxis, :]

print("Normalization completed.")
print("Normalized shape:", X_normalized.shape)


# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

print("\nCreating training and testing datasets...")

(
    X_train_full,
    X_test,
    y_train_full,
    y_test,
    snr_train_full,
    snr_test
) = train_test_split(
    X_normalized,
    y_encoded,
    snr_labels,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Train + Validation:", X_train_full.shape)
print("Test:", X_test.shape)


# ============================================================
# 9. TRAIN / VALIDATION SPLIT
# ============================================================

print("\nCreating training and validation datasets...")

(
    X_train,
    X_val,
    y_train,
    y_val,
    snr_train,
    snr_val
) = train_test_split(
    X_train_full,
    y_train_full,
    snr_train_full,
    test_size=0.125,
    random_state=42,
    stratify=y_train_full
)

print("\nFinal dataset shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


# ============================================================
# 10. DISPLAY FINAL SPLIT
# ============================================================

total_samples = len(X_normalized)

train_percent = len(X_train) / total_samples * 100
val_percent = len(X_val) / total_samples * 100
test_percent = len(X_test) / total_samples * 100

print("\n" + "=" * 60)
print("FINAL DATA SPLIT")
print("=" * 60)

print(f"Training:   {len(X_train)} samples ({train_percent:.2f}%)")
print(f"Validation: {len(X_val)} samples ({val_percent:.2f}%)")
print(f"Testing:    {len(X_test)} samples ({test_percent:.2f}%)")


# ============================================================
# 11. SAVE PROCESSED DATA
# ============================================================

print("\nSaving processed data...")

np.save(
    os.path.join(processed_data_dir, "X_train.npy"),
    X_train
)

np.save(
    os.path.join(processed_data_dir, "X_val.npy"),
    X_val
)

np.save(
    os.path.join(processed_data_dir, "X_test.npy"),
    X_test
)

np.save(
    os.path.join(processed_data_dir, "y_train.npy"),
    y_train
)

np.save(
    os.path.join(processed_data_dir, "y_val.npy"),
    y_val
)

np.save(
    os.path.join(processed_data_dir, "y_test.npy"),
    y_test
)

np.save(
    os.path.join(processed_data_dir, "snr_train.npy"),
    snr_train
)

np.save(
    os.path.join(processed_data_dir, "snr_val.npy"),
    snr_val
)

np.save(
    os.path.join(processed_data_dir, "snr_test.npy"),
    snr_test
)

np.save(
    os.path.join(processed_data_dir, "modulation_classes.npy"),
    label_encoder.classes_
)


# ============================================================
# 12. VERIFY SAVED FILES
# ============================================================

print("\nSaved files:")

for filename in sorted(os.listdir(processed_data_dir)):
    print("-", filename)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 60)

print(f"Total samples: {total_samples}")
print(f"Modulation classes: {len(label_encoder.classes_)}")
print(f"Input shape for CNN: {X_train.shape[1:]}")

print("\nCompleted:")
print("[✓] Dataset loaded")
print("[✓] IQ signals combined")
print("[✓] Labels created")
print("[✓] Labels encoded")
print("[✓] Signals normalized")
print("[✓] Training data created")
print("[✓] Validation data created")
print("[✓] Testing data created")
print("[✓] Processed data saved")

DATA PREPROCESSING STARTED

Dataset found:
E:\Major_Project\AI-Based-Intelligent-RF-Spectrum-Signal-Identification-System-main\data\dataset\RML2016.10a_dict.pkl

Loading RadioML 2016.10a dataset...
Dataset loaded successfully.

Number of modulation classes: 11
Modulation classes: ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']

Number of SNR levels: 20
SNR values: [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

Combining IQ signals...

Combined dataset shape:
X shape: (220000, 2, 128)
y shape: (220000,)
SNR shape: (220000,)

Label encoding:
0 -> 8PSK
1 -> AM-DSB
2 -> AM-SSB
3 -> BPSK
4 -> CPFSK
5 -> GFSK
6 -> PAM4
7 -> QAM16
8 -> QAM64
9 -> QPSK
10 -> WBFM

Class distribution:
8PSK: 20000
AM-DSB: 20000
AM-SSB: 20000
BPSK: 20000
CPFSK: 20000
GFSK: 20000
PAM4: 20000
QAM16: 20000
QAM64: 20000
QPSK: 20000
WBFM: 20000

Normalizing IQ signals...
Normalization completed.
Normalized shape: (220000, 2, 128)

Crea